DS-2002 Data Project 1 - Medical Appointments Data Mart

Overview: This notebook implements an ETL pipeline that extracts data from three different sources (MySQL, MongoDB, and a CSV file), transforms it, and loads it into a dimensional data mart in MySQL. The business process modeled is medical appointment scheduling, tracking interactions between patients, appointment slots, and dates.

Data Sources:

- MySQL (local): Used as the destination data warehouse (medical_dw)
- MongoDB Atlas (cloud): Stores appointment slot data (dim_slots)
- CSV File System: Stores patient data (patients.csv) and appointment data (appointments.csv)

Star Schema:
- Fact Table: fact_appointments
- Dimensions: dim_date, dim_patients, dim_slots, dim_age_groups

In [1]:
import os
import json
import numpy
import pandas as pd
import pymongo
import certifi
from sqlalchemy import create_engine, text

In [2]:
# MySQL connection
mysql_args = {
    "uid" : "nnguyen",
    "pwd" : "password",
    "hostname" : "localhost",
    "dbname" : "medical_dw"
}

# MongoDB Atlas connection
mongodb_args = {
    "user_name" : "nnguyen",
    "password" : "password!",     
    "cluster_name" : "cluster0",
    "cluster_subnet" : "5vzg5lm",
    "cluster_location" : "atlas",
    "db_name" : "medical_appointments"
}

In [3]:
# Connects to MySQL and returns query results as a DataFrame
def get_sql_dataframe(sql_query, **args):
    conn_str = f"mysql+pymysql://{args['uid']}:{args['pwd']}@{args['hostname']}/{args['dbname']}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()
    dframe = pd.read_sql(text(sql_query), connection)
    connection.close()
    return dframe

# Loads a DataFrame into a MySQL table, creating a PK if inserting
def set_dataframe(df, table_name, pk_column, db_operation, **args):
    conn_str = f"mysql+pymysql://{args['uid']}:{args['pwd']}@{args['hostname']}/{args['dbname']}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    db_connection = sqlEngine.connect()
    
    if db_operation == "insert":
        df.to_sql(table_name, con=db_connection, index=False, if_exists='replace')
        db_connection.execute(text(f"ALTER TABLE {table_name} ADD {pk_column} INT AUTO_INCREMENT PRIMARY KEY FIRST;"))
    elif db_operation == "update":
        df.to_sql(table_name, con=db_connection, index=False, if_exists='append')
    else:
        print("db_operation must be 'insert' or 'update'")
    
    db_connection.close()

# Returns a connected MongoDB client (Atlas or local)
def get_mongo_client(**args):
    if args["cluster_location"] == "atlas":
        connect_str = f"mongodb+srv://{args['user_name']}:{args['password']}@"
        connect_str += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net"
        client = pymongo.MongoClient(
            connect_str,
            tls=True,
            tlsAllowInvalidCertificates=True
        )
    elif args["cluster_location"] == "local":
        client = pymongo.MongoClient("mongodb://localhost:27017/")
    return client

# Queries a MongoDB collection and returns results as a DataFrame
def get_mongo_dataframe(mongo_client, db_name, collection, query):
    db = mongo_client[db_name]
    dframe = pd.DataFrame(list(db[collection].find(query)))
    dframe.drop(['_id'], axis=1, inplace=True)
    mongo_client.close()
    return dframe

# Loads JSON files into MongoDB collections
def set_mongo_collections(mongo_client, db_name, data_directory, json_files):
    db = mongo_client[db_name]
    for file in json_files:
        db.drop_collection(file)
        json_file = os.path.join(data_directory, json_files[file])
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
            file = db[file]
            result = file.insert_many(json_object)
    mongo_client.close()

Step 1 Create Destination Database:

Drop and recreate medical_dw database to ensure clean slate before loading.

In [4]:
conn_str = f"mysql+pymysql://{mysql_args['uid']}:{mysql_args['pwd']}@{mysql_args['hostname']}"
sqlEngine = create_engine(conn_str, pool_recycle=3600)
connection = sqlEngine.connect()

connection.execute(text("DROP DATABASE IF EXISTS `medical_dw`;"))
connection.execute(text("CREATE DATABASE `medical_dw`;"))

connection.close()

Step 2 Create and Populate dim_date:

Generate a date dimension covering the full range of appointment dates in the dataset (2014–2024). This allows time-based analysis across days, months, quarters, and years.

In [5]:
conn_str = f"mysql+pymysql://{mysql_args['uid']}:{mysql_args['pwd']}@{mysql_args['hostname']}/{mysql_args['dbname']}"
sqlEngine = create_engine(conn_str, pool_recycle=3600)
connection = sqlEngine.connect()

connection.execute(text("""
CREATE TABLE IF NOT EXISTS dim_date(
    date_key int NOT NULL,
    full_date date NULL,
    date_name char(11) NOT NULL,
    date_name_us char(11) NOT NULL,
    date_name_eu char(11) NOT NULL,
    day_of_week tinyint NOT NULL,
    day_name_of_week char(10) NOT NULL,
    day_of_month tinyint NOT NULL,
    day_of_year smallint NOT NULL,
    weekday_weekend char(10) NOT NULL,
    week_of_year tinyint NOT NULL,
    month_name char(10) NOT NULL,
    month_of_year tinyint NOT NULL,
    is_last_day_of_month char(1) NOT NULL,
    calendar_quarter tinyint NOT NULL,
    calendar_year smallint NOT NULL,
    calendar_year_month char(10) NOT NULL,
    calendar_year_qtr char(10) NOT NULL,
    fiscal_month_of_year tinyint NOT NULL,
    fiscal_quarter tinyint NOT NULL,
    fiscal_year int NOT NULL,
    fiscal_year_month char(10) NOT NULL,
    fiscal_year_qtr char(10) NOT NULL,
    PRIMARY KEY (date_key)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
"""))

connection.execute(text("DROP PROCEDURE IF EXISTS PopulateDateDimension;"))

connection.execute(text("""
CREATE PROCEDURE PopulateDateDimension(BeginDate DATETIME, EndDate DATETIME)
BEGIN
    DECLARE LastDayOfMon CHAR(1);
    DECLARE FiscalYearMonthsOffset INT;
    DECLARE DateCounter DATETIME;
    DECLARE FiscalCounter DATETIME;
    SET FiscalYearMonthsOffset = 6;
    SET DateCounter = BeginDate;
    WHILE DateCounter <= EndDate DO
        SET FiscalCounter = DATE_ADD(DateCounter, INTERVAL FiscalYearMonthsOffset MONTH);
        IF MONTH(DateCounter) = MONTH(DATE_ADD(DateCounter, INTERVAL 1 DAY)) THEN
            SET LastDayOfMon = 'N';
        ELSE
            SET LastDayOfMon = 'Y';
        END IF;
        INSERT INTO dim_date VALUES (
            (YEAR(DateCounter)*10000)+(MONTH(DateCounter)*100)+DAY(DateCounter),
            DateCounter,
            CONCAT(CAST(YEAR(DateCounter) AS CHAR(4)),'/',DATE_FORMAT(DateCounter,'%m'),'/',DATE_FORMAT(DateCounter,'%d')),
            CONCAT(DATE_FORMAT(DateCounter,'%m'),'/',DATE_FORMAT(DateCounter,'%d'),'/',CAST(YEAR(DateCounter) AS CHAR(4))),
            CONCAT(DATE_FORMAT(DateCounter,'%d'),'/',DATE_FORMAT(DateCounter,'%m'),'/',CAST(YEAR(DateCounter) AS CHAR(4))),
            DAYOFWEEK(DateCounter),
            DAYNAME(DateCounter),
            DAYOFMONTH(DateCounter),
            DAYOFYEAR(DateCounter),
            CASE DAYNAME(DateCounter) WHEN 'Saturday' THEN 'Weekend' WHEN 'Sunday' THEN 'Weekend' ELSE 'Weekday' END,
            WEEKOFYEAR(DateCounter),
            MONTHNAME(DateCounter),
            MONTH(DateCounter),
            LastDayOfMon,
            QUARTER(DateCounter),
            YEAR(DateCounter),
            CONCAT(CAST(YEAR(DateCounter) AS CHAR(4)),'-',DATE_FORMAT(DateCounter,'%m')),
            CONCAT(CAST(YEAR(DateCounter) AS CHAR(4)),'Q',QUARTER(DateCounter)),
            MONTH(FiscalCounter),
            QUARTER(FiscalCounter),
            YEAR(FiscalCounter),
            CONCAT(CAST(YEAR(FiscalCounter) AS CHAR(4)),'-',DATE_FORMAT(FiscalCounter,'%m')),
            CONCAT(CAST(YEAR(FiscalCounter) AS CHAR(4)),'Q',QUARTER(FiscalCounter))
        );
        SET DateCounter = DATE_ADD(DateCounter, INTERVAL 1 DAY);
    END WHILE;
END
"""))

connection.execute(text("CALL PopulateDateDimension('2014-01-01', '2024-12-31');"))
connection.commit()
connection.close()

Step 3 Load dim_patients from CSV (File System Source):

Extracts patient data from patients.csv, renames columns for clarity, removes duplicates, and loads into MySQL as dim_patients.

In [6]:
df_patients = pd.read_csv("patients.csv")

print(df_patients.shape)
print(df_patients.head())

(36697, 5)
   patient_id              name     sex         dob         insurance
0           1      Allison Hill  Female  1946-12-30   Mediflora Nexus
1           2      Nancy Rhodes  Female  1969-02-21  BioCrest Harmony
2           3   Angie Henderson  Female  1952-01-09  BioCrest Harmony
3           4    Colleen Wagner  Female  1981-01-28  BioCrest Harmony
4           5  Christina Santos  Female  1989-05-19     CurativeWhale


In [7]:
df_dim_patients = df_patients[['patient_id', 'name', 'sex', 'dob', 'insurance']].copy()

df_dim_patients.columns = ['patient_id', 'patient_name', 'sex', 'date_of_birth', 'insurance']

df_dim_patients = df_dim_patients.drop_duplicates()

print(df_dim_patients.shape)
print(df_dim_patients.head())

set_dataframe(df_dim_patients, "dim_patients", "patient_key", "insert", **mysql_args)

(36697, 5)
   patient_id      patient_name     sex date_of_birth         insurance
0           1      Allison Hill  Female    1946-12-30   Mediflora Nexus
1           2      Nancy Rhodes  Female    1969-02-21  BioCrest Harmony
2           3   Angie Henderson  Female    1952-01-09  BioCrest Harmony
3           4    Colleen Wagner  Female    1981-01-28  BioCrest Harmony
4           5  Christina Santos  Female    1989-05-19     CurativeWhale


Step 4 Load dim_slots from MongoDB (NoSQL Source):

Converts slots.csv to JSON format, loads it into MongoDB Atlas as a collection,then extracts it back out and loads it into MySQL as dim_slots.

In [8]:
# Note: slots.csv is trimmed to 10,000 rows to avoid MongoDB Atlas 
# free tier connection timeout when uploading large files
df_slots = pd.read_csv("slots.csv")
df_slots = df_slots.head(10000)

print(df_slots.shape)
print(df_slots.head())

slots_json_path = "slots.json"
df_slots.to_json(slots_json_path, orient='records', indent=2)

(10000, 4)
   slot_id appointment_date appointment_time  is_available
0        1       2015-01-01         08:00:00         False
1        2       2015-01-01         08:15:00         False
2        3       2015-01-01         08:30:00         False
3        4       2015-01-01         08:45:00         False
4        5       2015-01-01         09:00:00         False


In [9]:
import math

client = get_mongo_client(**mongodb_args)
db = client[mongodb_args["db_name"]]

# Drop existing collection
db.drop_collection("slots")

# Read slots json
with open(os.path.join(os.getcwd(), "slots.json"), 'r') as f:
    slots_data = json.load(f)

# Upload in chunks of 1000 to avoid connection timeout
chunk_size = 1000
total_chunks = math.ceil(len(slots_data) / chunk_size)

for i in range(total_chunks):
    chunk = slots_data[i * chunk_size:(i + 1) * chunk_size]
    db["slots"].insert_many(chunk)
    if i % 10 == 0:
        print(f"Uploaded chunk {i+1}/{total_chunks}")

client.close()

Uploaded chunk 1/10


In [10]:
client = get_mongo_client(**mongodb_args)
df_slots_mongo = get_mongo_dataframe(client, mongodb_args["db_name"], "slots", {})

df_dim_slots = df_slots_mongo[['slot_id', 'appointment_date', 'appointment_time', 'is_available']].copy()
df_dim_slots = df_dim_slots.drop_duplicates()

print(df_dim_slots.shape)
print(df_dim_slots.head())

set_dataframe(df_dim_slots, "dim_slots", "slot_key", "insert", **mysql_args)

(10000, 4)
   slot_id appointment_date appointment_time  is_available
0        1       2015-01-01         08:00:00         False
1        2       2015-01-01         08:15:00         False
2        3       2015-01-01         08:30:00         False
3        4       2015-01-01         08:45:00         False
4        5       2015-01-01         09:00:00         False


Step 5 Load dim_age_groups from Appointments Data:

Extracts the distinct age group values from appointments.csv and loads them as a dimension table for demographic analysis.

In [11]:
df_appointments = pd.read_csv("appointments.csv")

df_dim_age_groups = df_appointments[['age_group']].drop_duplicates().reset_index(drop=True)
df_dim_age_groups.columns = ['age_group']

print(df_dim_age_groups.shape)
print(df_dim_age_groups)

set_dataframe(df_dim_age_groups, "dim_age_groups", "age_group_key", "insert", **mysql_args)

(16, 1)
   age_group
0      35-39
1      80-84
2      75-79
3      70-74
4      50-54
5      25-29
6      30-34
7        90+
8      65-69
9      60-64
10     55-59
11     85-89
12     45-49
13     40-44
14     15-19
15     20-24


Step 6: Build and Load fact_appointments (SQL Source):

Builds the central fact table by merging surrogate keys from all dimension tables onto the raw appointments data. Drops source columns, keeping only foreign keys and measurable facts like duration, waiting time, status, scheduling.

In [12]:
df_fact = df_appointments.copy()

df_fact['date_key'] = pd.to_datetime(df_fact['appointment_date']).dt.strftime('%Y%m%d').astype(int)

df_patient_keys = get_sql_dataframe("SELECT patient_key, patient_id FROM medical_dw.dim_patients;", **mysql_args)
df_fact = df_fact.merge(df_patient_keys, on='patient_id', how='left')

df_slot_keys = get_sql_dataframe("SELECT slot_key, slot_id FROM medical_dw.dim_slots;", **mysql_args)
df_fact = df_fact.merge(df_slot_keys, on='slot_id', how='left')

df_age_keys = get_sql_dataframe("SELECT age_group_key, age_group FROM medical_dw.dim_age_groups;", **mysql_args)
df_fact = df_fact.merge(df_age_keys, on='age_group', how='left')

df_fact_appointments = df_fact[[
    'appointment_id',
    'patient_key',
    'slot_key',
    'date_key',
    'age_group_key',
    'scheduling_interval',
    'status',
    'appointment_duration',
    'waiting_time'
]].copy()

print(df_fact_appointments.shape)
print(df_fact_appointments.head())

set_dataframe(df_fact_appointments, "fact_appointments", "appointment_key", "insert", **mysql_args)

(111488, 9)
   appointment_id  patient_key  slot_key  date_key  age_group_key  \
0             138         8790       1.0  20150101              1   
1             146         5764      54.0  20150101              2   
2              21         6514      55.0  20150101              3   
3             233         5296      56.0  20150101              1   
4              90         7915      57.0  20150101              4   

   scheduling_interval          status  appointment_duration  waiting_time  
0                    4  did not attend                   NaN           NaN  
1                    3  did not attend                   NaN           NaN  
2                   15        attended                   5.2           1.2  
3                    1        attended                  28.9           1.1  
4                    6       cancelled                   NaN           NaN  


Step 7 Verify Data Mart with SQL Queries:

Demonstrates functionality of the data mart by running analytical queries that joins the fact table with multiple dimension tables and performs aggregations.

In [13]:
sql_query1 = """
SELECT p.insurance, d.calendar_year, COUNT(*) AS total_appointments,
       AVG(f.appointment_duration) AS avg_duration_mins
FROM medical_dw.fact_appointments f
JOIN medical_dw.dim_patients p ON f.patient_key = p.patient_key
JOIN medical_dw.dim_date d ON f.date_key = d.date_key
GROUP BY p.insurance, d.calendar_year
ORDER BY total_appointments DESC;
"""

df_result1 = get_sql_dataframe(sql_query1, **mysql_args)
print("Query 1: Appointments by Insurance Provider and Year")
print(df_result1.head(10))

sql_query2 = """
SELECT ag.age_group, d.month_name, d.calendar_year, 
       COUNT(*) AS total_appointments,
       SUM(f.waiting_time) AS total_waiting_time
FROM medical_dw.fact_appointments f
JOIN medical_dw.dim_age_groups ag ON f.age_group_key = ag.age_group_key
JOIN medical_dw.dim_date d ON f.date_key = d.date_key
GROUP BY ag.age_group, d.month_name, d.calendar_year, d.month_of_year
ORDER BY d.calendar_year, d.month_of_year;
"""

df_result2 = get_sql_dataframe(sql_query2, **mysql_args)
print("Query 2: Appointments by Age Group and Month")
print(df_result2.head(10))

Query 1: Appointments by Insurance Provider and Year
        insurance  calendar_year  total_appointments  avg_duration_mins
0  Vitalynx Orbit           2021                3637          17.261154
1  Vitalynx Orbit           2022                3629          17.508909
2  Vitalynx Orbit           2023                3626          17.644770
3  Vitalynx Orbit           2019                3613          17.662306
4  Vitalynx Orbit           2020                3579          17.300107
5  Vitalynx Orbit           2018                3551          17.776584
6  Vitalynx Orbit           2016                3550          17.245009
7  Vitalynx Orbit           2015                3541          17.459167
8  Vitalynx Orbit           2017                3541          17.492000
9  Vitalynx Orbit           2024                3273          17.897426
Query 2: Appointments by Age Group and Month
  age_group month_name  calendar_year  total_appointments  total_waiting_time
0     35-39    January          

Deployment Strategy:

- MySQL: Runs locally on the user's machine (localhost)
- MongoDB: Hosted on MongoDB Atlas (free tier cloud cluster)
- Python/Jupyter: Run locally using Jupyter Notebook
- Data files: CSV files stored locally in the same directory as the notebook
- 
How to Run:
1. Make sure MySQL is running locally
2. Make sure all CSV files are in the same folder as this notebook
3. Update your MySQL and MongoDB credentials in Cell 2
4. Run all cells from top to bottom
5. Required Python Packages: pandas, pymongo, sqlalchemy, pymysql, certifi